<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/evaluation/02_Hyperparameter_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================
# [Evaluation 02] Hyperparameter Tuning with Optuna
# ==================================================================
import os
import sys
import subprocess

# ... (01번 노트북과 동일한 Github 클론 및 모듈 셋업 코드 적용) ...

# Optuna 및 시각화 도구 설치
!pip install -q optuna plotly nest_asyncio

import nest_asyncio
nest_asyncio.apply()
print("\n🎉 Optuna 튜닝 환경 셋업 완료.")


In [ ]:
import optuna
import asyncio
from pathlib import Path
from src.evaluation import run_transcription_evaluation
import numpy as np

# 튜닝에 사용할 소규모 Validation 데이터셋 (빠른 탐색을 위해 3~5곡만 사용)
LOCAL_EVAL_DIR = Path("/content/slakh_eval")
val_tracks = [d for d in LOCAL_EVAL_DIR.iterdir() if d.is_dir()][:3]

print(f"🔍 최적화를 위해 {len(val_tracks)}개의 트랙을 사용합니다.")

async def objective(trial):
    """
    Optuna가 파라미터 조합을 던져주면, 그 조합으로 채보를 수행하고 평균 F1-Score를 반환합니다.
    """
    # 1. 튜닝할 하이퍼파라미터 탐색 공간 정의 (예시)
    # 실제 src 내부 로직에 전달될 파라미터들로 교체하세요!
    p_onset_tolerance = trial.suggest_float('onset_tolerance', 0.05, 0.15, step=0.01)

    # 예: src 로직 어딘가에 들어갈 파라미터를 임의로 조절한다고 가정
    # p_viterbi_penalty = trial.suggest_float('viterbi_penalty', 0.1, 2.0)
    # p_debounce_ms = trial.suggest_int('debounce_ms', 10, 100)

    f1_scores = []

    for track in val_tracks:
        ref_midi = str(track / "bass_gt.mid")
        input_audio = str(track / "bass_gt.wav") # Isolated 모드로 순수 알고리즘만 평가

        try:
            # 평가 함수 호출 시 trial 파라미터 주입
            metrics = await run_transcription_evaluation(
                ref_midi_path=ref_midi,
                audio_path=input_audio,
                is_isolated=True,
                onset_tolerance=p_onset_tolerance
            )
            f1_scores.append(metrics.get('Onset_Pitch_F1', 0))
        except Exception as e:
            pass

    # 평균 F1-Score 반환 (이 값을 최대화하는 것이 목표)
    return np.mean(f1_scores) if f1_scores else 0.0

def run_optuna():
    # 방향 설정: F1-Score는 높을수록 좋으므로 'maximize'
    study = optuna.create_study(direction='maximize', study_name="Bass_Transcription_Tuning")

    # 20번의 실험(Trial)을 수행하여 최적의 파라미터 탐색
    study.optimize(lambda t: asyncio.run(objective(t)), n_trials=20)
    return study

study = run_optuna()


In [ ]:
# ==================================================================
# [Results] 최적 파라미터 출력 및 최적화 과정 시각화
# ==================================================================
import optuna.visualization as vis

print("\n" + "="*40)
print("🏆 최적화 탐색 완료!")
print("="*40)
print(f"최고 F1-Score: {study.best_value * 100:.2f}%")
print("최적 파라미터 조합:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

# 1. 파라미터 중요도 시각화 (어떤 파라미터가 점수에 가장 큰 영향을 주었는가?)
fig_importances = vis.plot_param_importances(study)
fig_importances.show()

# 2. 최적화 히스토리 시각화 (실험을 거듭하며 점수가 어떻게 올랐는가?)
fig_history = vis.plot_optimization_history(study)
fig_history.show()
